<a href="https://colab.research.google.com/github/thalitadru/ml-class-epf/blob/main/LabAssignmentRNNsToyTimeSeries_complete.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Processing Sequences Using RNNs (and CNNs)

*Credits:* Based on code written by [A. Géron](https://colab.research.google.com/github/ageron/handson-ml2/blob/master/15_processing_sequences_using_rnns_and_cnns.ipynb#scrollTo=AiINDLJHVNep) for his "Hands-on ML" book. Code realeased under MIT license.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In this activity we are going to explore different ways of modelling sequential data. We will try out simple baseline models as well as 1D CNNs and different RNNs to **forecast the next step in a time series**.

## Loading a real-world ridership dataset
We will use the Chicago Transit Authority daily ridership dataset used in the latest edition of Hands-On Machine Learning. We will forecast rail ridership from previous observations.

In [ ]:
import tensorflow as tf
dir = tf.keras.utils.get_file("ridership.tgz","https://github.com/ageron/data/raw/main/ridership.tgz",cache_dir=".",extract=True)
from pathlib import Path
path = Path(dir) / "ridership/CTA_-_Ridership_-_Daily_Boarding_Totals.csv"
df = pd.read_csv(path, parse_dates=["service_date"])
df.columns=["date","day_type","bus","rail","total"]
df=df.sort_values("date").set_index("date")
df=df.drop("total",axis=1).drop_duplicates()


In [ ]:
n_steps = 56
ahead = 14
rail_train = df["rail"]["2016-01":"2018-12"] / 1e6
rail_valid = df["rail"]["2019-01":"2019-05"] / 1e6
rail_test = df["rail"]["2019-06":] / 1e6


Notice the shape of the series array:


In [ ]:
"Rail series length:", len(rail_train), len(rail_valid), len(rail_test)

The meaning of each dimension is `[samples, time steps, sequence element size]`. 
Here each series element is a single scalar because we are making a univariate prediction.
If we were modeling mutiple time series at once (eg. temperature and humidity) this would be a multi-variate prediction and our sequence element size would be 2.

### Creating predicion targets
We want to predict one step further into the series. To do so , we can take the last time step of a series as the regression target, while using all the previous time steps as inputs.

We also want to split our data into train, validation and test sets as usual.

In [ ]:
def make_windows(series, window=56):
    X,y=[],[]
    arr=series.to_numpy()
    for i in range(len(arr)-window):
        X.append(arr[i:i+window])
        y.append(arr[i+window])
    return np.array(X)[...,np.newaxis], np.array(y)[...,np.newaxis]
X_train_one,y_train_one = make_windows(rail_train,n_steps)
X_valid_one,y_valid_one = make_windows(rail_valid,n_steps)
X_test_one,y_test_one = make_windows(rail_test,n_steps)


In [ ]:
X_train_one.shape, y_train_one.shape

### Visualizing the data
This function plots the first series samples from a given batch, along with the next predicted value.

In [ ]:
def plot_series(series, y=None, y_pred=None, x_label="$t$", y_label="$x(t)$", legend=True):
    plt.plot(series, ".-")
    if y is not None:
        plt.plot(n_steps, y, "bo", label="Target")
    if y_pred is not None:
        plt.plot(n_steps, y_pred, "rx", markersize=10, label="Prediction")
    plt.grid(True)
    if x_label:
        plt.xlabel(x_label, fontsize=16)
    if y_label:
        plt.ylabel(y_label, fontsize=16, rotation=0)
    plt.xlim([0, n_steps + 1])
    if legend and (y or y_pred):
        plt.legend(fontsize=14, loc="upper left")

fig, axes = plt.subplots(nrows=1, ncols=3, sharey=True, figsize=(12, 4))
for col in range(3):
    plt.sca(axes[col])
    plot_series(X_valid_one[col, :, 0], y_valid_one[col, 0],
                y_label=("$x(t)$" if col==0 else None),
                legend=(col == 0))
plt.show()

## Prediction task 1 : forecast one time step

We want to predict a single continuous value for each input sequence. This can be seen as a sequence-to-vector regression task. As seen in class, a suitable loss function is the `keras.losses.mean_squared_error` (as derived by MLE under a Gaussian assumption on the prediciton targets).


As we try out different models, keep the models and their validation performances. In the end you will compare their performances in validation to pick the best model (which will be applied to the test set).


### Simple baseline models

Before trying complex models, it is important to set up a baseline performance obtained with a simple model. This way we will only be interested in the more complex models if they do better than our baseline.

#### Baseline 1: Naive baseline

Our first baseline will follow a very simple rule:
Ridership exhibits a strong weekly seasonality.
As a simple baseline, we use a seasonal naive forecast and predict that the next value will be equal to the value observed one week earlier.

This is very simple to implement: our predictions `y_pred` will simply be equal to step `-7` in the `X` array.



##### TODO: write the naive baseline code bellow
Compute the naive predctions on the validation set and save them to a `y_pred` array.

In [ ]:
#TODO


In [ ]:
#SOLUTION
y_pred = X_valid_one[:, -7]
val_loss = np.mean(keras.losses.mse(y_valid_one, y_pred))
val_loss

Here is the first series in the validation set with its predicted value:

In [ ]:
plot_series(X_valid_one[0, :, 0], y_valid_one[0, 0], y_pred[0, 0])

#### Baseline 2: linear baseline

Our second basline model will be a linear regression. We can implement it using keras `Sequential` API by simply using a `Dense` layer with no activation functions.



##### TODO: Declare and train the linear model
 - write a linear regression model using a Dense layer
 - Don't forget to use `Flatten` and define the expected input shape using the keyword argument `input_shape`
 - Use the Adam optimizer with the default learning rate
 - Train it it for 20 epochs
 - Don't forget to include your validation data
 - Check the learning curves and the performance on the validation set


In [ ]:
np.random.seed(42)
tf.random.set_seed(42)

#TODO declare and train your model here
model = ...

model.compile(...)

history = model.fit(...)

In [ ]:
#SOLUTION
np.random.seed(42)
tf.random.set_seed(42)

model = keras.models.Sequential([
    keras.layers.Input([n_steps,1]),
    keras.layers.Flatten(),
    keras.layers.Dense(1)
])

model.compile(loss="mse", optimizer="adam")
history = model.fit(X_train_one, y_train_one, epochs=20,
                    validation_data=(X_valid_one, y_valid_one))

In [ ]:
# saving model stats for later
pred_one_models = {}
pred_one_models['linear']= {'model': model, 'history': history}

In [ ]:
pd.DataFrame(history.history).plot()

In [ ]:
model.evaluate(X_valid_one, y_valid_one)

In [ ]:
y_pred = model.predict(X_valid_one)
plot_series(X_valid_one[0, :, 0], y_valid_one[0, 0], y_pred[0, 0])

### Classical forecasting baseline
So far our baselines only relied on recent observations.
A classical approach for forecasting time series is to explicitly model trend, autoregressive effects and seasonality. One such model is Seasonal ARIMA (SARIMA).

#### Baseline 3: SARIMA
Here we use a SARIMA model with a weekly seasonal period (7 days), which is particularly relevant for public transport ridership data.

Unlike the neural-network models above, SARIMA is trained directly on the time series rather than on sliding-window inputs.

In [ ]:
from statsmodels.tsa.arima.model import ARIMA

In [ ]:
#TODO
# Train a SARIMA model on the rail_train series
# Use:
#   order=(1,0,0)
#   seasonal_order=(0,1,1,7)
#
# Then generate predictions for all dates in the validation period
# and compute the MSE.

In [ ]:
#SOLUTION
sarima_model = ARIMA(
    rail_train,
    order=(1, 0, 0),
    seasonal_order=(0, 1, 1, 7)
)
# Fitting model parameters
sarima_results = sarima_model.fit()

In [ ]:
# Predicting on the validation set in order to evaluate the model
sarima_pred = sarima_results.forecast(steps=len(rail_valid))
# calculating MSE over the validation set
val_loss = np.mean(np.square(rail_valid.to_numpy() - sarima_pred.to_numpy()))
print(f"SARIMA validaiton MSE:{val_loss}")

Here is some code to visualise the results

In [ ]:
plt.figure(figsize=(10,4))

plt.plot(rail_valid.index, rail_valid, label="Actual")

plt.plot(sarima_pred.index, sarima_pred, label="SARIMA")

plt.legend()
plt.grid(True)
plt.title("SARIMA validation forecasts")
plt.show()

Let's save the results for later comparison:

In [ ]:
pred_one_models["SARIMA"] = {
    "model": sarima_results,
    "val_loss": val_loss
}

### Recurrent models

#### Model 1: Simple RNN
Our first recurrent model will use simple recurrent units. You can implement it using the built-in `keras.layers.SimpleRNN`.

Here again we will specify the expected input shape (ignoring the batch dimension). We want the time-steps dimension to be variable: thus we set it to `None`. The sequence element dimension continues to be 1.

##### TODO: train the simple RNN model
 - Use the Adam optimizer with the a learning rate of 0.02
 - Train it it for 20 epochs
 - Don't forget to include your validation data
 - Check the learning curves and the performance on the validation set

In [ ]:
np.random.seed(42)
tf.random.set_seed(42)

model = keras.models.Sequential([
    keras.layers.Input([None,1]),
    keras.layers.SimpleRNN(1)
])

#TODO train the model here


In [ ]:
#SOLUTION
np.random.seed(42)
tf.random.set_seed(42)

model = keras.models.Sequential([
    keras.layers.Input([None,1]),
    keras.layers.SimpleRNN(1)
])


model.compile(loss="mse", optimizer="adam")
model.optimizer.learning_rate = 2e-2
history = model.fit(X_train_one, y_train_one, epochs=20,
                    validation_data=(X_valid_one, y_valid_one))

In [ ]:
# saving model stats for later
pred_one_models['simpleRNN']= {'model': model, 'history': history}

In [ ]:
pd.DataFrame(history.history).plot()

In [ ]:
model.evaluate(X_valid_one, y_valid_one)

In [ ]:
y_pred = model.predict(X_valid_one)
plot_series(X_valid_one[0, :, 0], y_valid_one[0, 0], y_pred[0, 0])
plt.show()

#### Model 2: Deep simple RNN
This time we will use multiple recurrent layers in our model. The final output will be computed with a dense layer.


##### Chaining two RNN layers
Notice that while recurrent layers expect sequence inputs with 3 dimensions (samples, time, element size), they output by default 2 D data as in `[samples, predicted element size]`. 

If you chain two RNN layers, the first one needs yield sequence-shaped outputs (with the 3 dimensions) so that it is compatible with the second layer. In this case, the layer needs to be declared with the keywork argument `return_sequences=True`.

##### TODO: Create and train a Deep RNN with 2 hidden recurrent layers
 - Use two simple RNN layers with 20 units
 - Compute the final output doing a linear read-out (use a Dense layer with no activation)
 - Use the Adam optimizer with the default learning rate
 - Train it it for 20 epochs
 - Don't forget to include your validation data
 - Check the learning curves and the performance on the validation set

In [ ]:
np.random.seed(42)
tf.random.set_seed(42)

#TODO declare and train your model

In [ ]:
#SOLUTION
np.random.seed(42)
tf.random.set_seed(42)

model = keras.models.Sequential([
    keras.layers.Input([None, 1]),
    keras.layers.SimpleRNN(20, return_sequences=True),
    keras.layers.SimpleRNN(20),
    keras.layers.Dense(1)
])

model.compile(loss="mse", optimizer="adam")
history = model.fit(X_train_one, y_train_one, epochs=20,
                    validation_data=(X_valid_one, y_valid_one))

In [ ]:
# saving model stats for later
pred_one_models['DeepRNN']= {'model': model, 'history': history}

In [ ]:
pd.DataFrame(history.history).plot()

In [ ]:
model.evaluate(X_valid_one, y_valid_one)

In [ ]:
y_pred = model.predict(X_valid_one)
plot_series(X_valid_one[0, :, 0], y_valid_one[0, 0], y_pred[0, 0])

### TODO: Final comparison and model choice
Compare the compare performances of different models in validation to pick the best model. 
Apply it to the test set and evaluate it: is the performance close to what you got in the validation set?

In [ ]:
#TODO

In [ ]:
pred_one_models

In [ ]:
#SOLUTION
data = []
for estim, log in pred_one_models.items():
    try:
        val_loss = log['history'].history['val_loss'][-1]
    except:
        val_loss = log['val_loss']
    data.append({'Model': estim, 
                'Validation loss': val_loss})
df = pd.DataFrame(data)
df

In [ ]:
# SOLUTION if SARIMA wins

sarima_retrain = ARIMA(
    pd.concat([rail_train, rail_valid]),
    order=(1, 0, 0),
    seasonal_order=(0, 1, 1, 7)
)
sarima_results = sarima_retrain.fit()

sarima_test_pred = sarima_results.forecast(steps=len(rail_test))

test_loss = np.mean(np.square(rail_test.to_numpy() - sarima_test_pred.to_numpy()))

test_loss

#SOLUTION
You need to inspect the validation results table above and select the model with lowest error.


**In this run: best model was the deep RNN**
Here is it's performance on the test set.

In [ ]:
#SOLUTION
y_pred = model.predict(X_test_one)
plot_series(X_test_one[0, :, 0], y_test_one[0, 0], y_pred[0, 0])

In [ ]:
#SOLUTION
model = pred_one_models['DeepRNN']['model']
model.evaluate(X_test_one, y_test_one)

## Prediction task 2: forecast multiple time steps

We now want to predict the next 14 days instead of only the next day.

We will compare three approaches:

1. **Recursive forecasting**: repeatedly apply a one-step model.
2. **Direct forecasting (Seq2Vec)**: predict all 14 days at once.
3. **Seq2Seq forecasting**: predict a 14-day horizon at every input time step.

The three methods use different target representations and produce different output shapes.
| Method    | Training Target shape | Output shape                  | Evaluation target shape |
|-----------|-----------------------|-------------------------------|-------------------------|
| Recursive | [batch, 1]            | [batch, 1] repeatedly applied | [batch, 14]             |
| Seq2Vec   | [batch, 14]           | [batch, 14]                   | [batch, 14]             |
| Seq2Seq   | [batch, 56, 14]       | [batch, 56, 14]               | [batch, 56, 14]         |

#### Why compare different methods ?
These methods illustrate a common trade-off in forecasting:

- Recursive forecasting reuses a simple one-step model but may accumulate prediction errors.
- Seq2Vec forecasting predicts the whole horizon directly.
- Seq2Seq forecasting provides more training signals by producing predictions at every time step.


### Method 1: Recursive forecasting with a one-step model


We'll reuse the one-step forecasting model trained earlier and apply it recursively to predict the next 14 days.

<div class="alert alert-info">
💡This approach is attractive because we can <b>reuse an already-trained one-step model</b> without changing its architecture.
</div>

In [ ]:
X = X_valid_one.copy()
model = pred_one_models['DeepRNN']['model']
for step_ahead in range(ahead):
    y_pred_one = model.predict(X)[:, np.newaxis, :]
    X = np.concatenate([X, y_pred_one], axis=1)

Y_pred = X[:, -ahead:, 0]

In [ ]:
Y_pred.shape

#### Evaluation targets for recursive forecasting

To evaluate a 14-day recursive forecast, we need the true 14 future observations associated with each validation window.

These targets are only used for evaluation. They are not used to train the one-step model.

In [ ]:
# Build the true 14-day future horizon associated with each
# validation window.

Y_valid_recursive = np.column_stack([
    rail_valid.to_numpy()[n_steps + step_ahead :
                          n_steps + step_ahead + len(X_valid_one)]
    for step_ahead in range(ahead)
])

Y_valid_recursive.shape

#### TODO: check the loss value (MSE) for this method

In [ ]:
#TODO

In [ ]:
#SOLUTION
np.mean(keras.metrics.mse(Y_valid_recursive, Y_pred))

#### Plot multiple forecasts
Use this plotting function to visualize your input series and the output predictions.

**Attention**: Reshape the outputs of your models acordingly:
If you model outputs 2D arrays with shape [batch size, time steps], you can use `Y[..., np.newaxis]` or `np.expand_dims(Y, shape=-1) ` to make them compatible with this function.

In [ ]:
def plot_multiple_forecasts(X, Y, Y_pred):
    # X should be a rank 3 array with shapes [batch size, sequence length, 1]
    # Y and y_pred should also be rank 3 with shape [0, sequence lenght, 1]
    n_steps = X.shape[1]
    ahead = Y.shape[1]
    plot_series(X[0, :, 0])
    plt.plot(np.arange(n_steps, n_steps + ahead), Y[0, :, 0], "bo-", label="Actual")
    plt.plot(np.arange(n_steps, n_steps + ahead), Y_pred[0, :, 0], "rx-", label="Forecast", markersize=10)
    plt.legend(fontsize=14)


In [ ]:
#Example
plot_multiple_forecasts(X_valid_one, Y_valid_recursive[..., np.newaxis], Y_pred[..., np.newaxis])

#### TODO: Baselines: naive and linear prediction

Implement the naive and the linear baselines folowing this method to predict the next 14 steps. Compare their performances to the previous RNN model.

In [ ]:
#SOLUTION
# Naive:
# Repeat last value [ahead] times for each sample

Y_pred_naive = np.repeat(X_valid_one[:, -7, :], ahead, axis=1 ) 
 
# # Use the model to predict the next 14 steps
# Linear
X = X_valid_one.copy()
model_lin = pred_one_models['linear']['model']

for step_ahead in range(ahead):
    y_pred_one = model_lin.predict(X[:,-56:,:])[:, np.newaxis, :]
    X = np.concatenate([X, y_pred_one], axis=1)
Y_pred_linear = X[:, -ahead:, 0]
 
mse_naive = np.mean(np.square(Y_valid_recursive - Y_pred_naive))
mse_linear = np.mean(np.square(Y_valid_recursive - Y_pred_linear))
mse_drnn = np.mean(np.square(Y_valid_recursive - Y_pred)) # Y_pred from the previous RNN prediction
 
print(f"Naive Baseline MSE: {mse_naive}")
print(f"Linear Baseline MSE: {mse_linear}")
print(f"DRNN Model MSE: {mse_drnn}")
 
plot_multiple_forecasts(X_valid_one, Y_valid_recursive[..., np.newaxis], Y_pred_naive[..., np.newaxis])
plt.show()
 
plot_multiple_forecasts(X_valid_one, Y_valid_recursive[..., np.newaxis], Y_pred_linear[..., np.newaxis])
plt.show()

### Method 2A: Predicting 14 steps at once at the end (seq2vec)

In this section the model processes the entire input sequence before producing a single output vector containing all 14 future predictions.

This remains a sequence-to-vector task because only a single output is produced after processing the whole input sequence.

<div class="alert alert-info">
⏱️Unlike recursive forecasting, all 14 future values are predicted in a single forward pass.
</div>

#### Framing data and targets for multi-step forecasting

For the remaining methods, each input window is paired with the next 14 days:

Input:
[t−55, ..., t]

Target:
[t+1, ..., t+14]

This target representation allows the model to learn the entire forecasting horizon directly.

In [ ]:
ahead = 14

def make_multi_step_windows(series, window=56, horizon=14):
    arr = series.to_numpy()

    X = []
    Y = []

    for i in range(len(arr) - window - horizon + 1):
        X.append(arr[i:i+window])
        Y.append(arr[i+window:i+window+horizon])

    return (
        np.array(X)[..., np.newaxis],
        np.array(Y)
    )


In [ ]:

X_train_multi, Y_train_multi = make_multi_step_windows(
    rail_train,
    window=n_steps,
    horizon=ahead
)

X_valid_multi, Y_valid_multi = make_multi_step_windows(
    rail_valid,
    window=n_steps,
    horizon=ahead
)

X_test_multi, Y_test_multi = make_multi_step_windows(
    rail_test,
    window=n_steps,
    horizon=ahead
)

### Naive prediction
Here we apply the same naive method as before to obtain predictions.

In [ ]:
# take the -7 time step value, and repeat it [ahead] times
# shape should be (n_samples, ahead)
Y_naive_pred = np.repeat(X_valid_multi[:, -7:], ahead, axis=1)
val_loss = np.mean(keras.metrics.mse(Y_valid_multi, Y_naive_pred))
val_loss

### Example with a Linear model

In [ ]:
np.random.seed(42)
tf.random.set_seed(42)

model = keras.models.Sequential([
    keras.layers.Input([n_steps, 1]),
    keras.layers.Flatten(),
    keras.layers.Dense(ahead)
])

model.compile(loss="mse", optimizer="adam")
history = model.fit(X_train_multi, Y_train_multi, epochs=20,
                    validation_data=(X_valid_multi, Y_valid_multi))

In [ ]:
# saving model stats for later
pred_multi_models = {}
pred_multi_models['linear']= {'model': model, 'history': history}

In [ ]:
pd.DataFrame(history.history).plot()

In [ ]:
Y_pred_multi = model.predict(X_valid_multi)
plot_multiple_forecasts(X_valid_multi, Y_valid_multi[..., np.newaxis], Y_pred_multi[..., np.newaxis])

#### Model 1: Deep simple RNN with output prediction at the end

Let's create an RNN that predicts all 14 next values at once. To do that, all you need is to change the size of the final `Dense` layer to 14.

##### TODO: build and train the new RNN model with output size = 14
- Use two simple RNN layers with 20 units
- Compute the final output doing a linear read-out (use a Dense layer with no activation)
- Use the Adam optimizer with the default learning rate
- Train it it for 20 epochs
- Don't forget to include your validation data
- Check the learning curves and the performance on the validation set

In [ ]:
np.random.seed(42)
tf.random.set_seed(42)

#TODO declare and train your model here



In [ ]:
#SOLUTION
np.random.seed(42)
tf.random.set_seed(42)

model = keras.models.Sequential([
    keras.layers.SimpleRNN(20, return_sequences=True, input_shape=[None, 1]),
    keras.layers.SimpleRNN(20),
    keras.layers.Dense(ahead)
])


model.compile(loss="mse", optimizer="adam")
history = model.fit(X_train_multi, Y_train_multi, epochs=20,
                    validation_data=(X_valid_multi, Y_valid_multi))

In [ ]:
# saving model stats for later
pred_multi_models['DeepRNN_seq2vec']= {'model': model, 'history': history}

In [ ]:
pd.DataFrame(history.history).plot()

In [ ]:
model.evaluate(X_valid_multi, Y_valid_multi)

In [ ]:
Y_pred_multi = model.predict(X_valid_multi)
plot_multiple_forecasts(X_valid_multi, Y_valid_multi[..., np.newaxis], Y_pred_multi[..., np.newaxis])

### Method 2B: Sequence-to-Sequence Forecasting (Seq2Seq)

Now let's create RNN models that predict the next 14 steps at each time step. 

This means that at t=0, the model predicts a vector containing the next 14 time steps. 
Then at t=1, it predicts another vector containing the next 14 future time steps. 
A crucial difference here is that for each time step we expect a prediction, while before we only cared about the last output. 
This implies that the loss will contain a term for each time step, not just the last. 
This way, during backpropagation, each time step contributes directly to the loss.
As a result, gradients originate from many output locations instead of only the final one.
In practice, this often provides a stronger training signal and may **stabilize or accelerate training**.



**Note:** In this model and all others used here, predictions for time $T$ only take into account past series values, i.e. where $t < T$. This defines them as a *causal* models. 

#### Framing seq2seq data and prediction targets
Instead of generating a single 14-day forecast at the end of the sequence, we generate a 14-day forecast at every time step.

Here is a more visual example for `ahead=3`:

```
Input window

x1 x2 x3 x4 x5 ...

Targets

at t=1 -> [x2 x3 x4]
at t=2 -> [x3 x4 x5]
at t=3 -> [x4 x5 x6]
...
```
This is why the target tensor now has three dimensions:
`[batch size, sequence length, forecast horizon]`.

In [ ]:
def make_seq2seq_windows(series, window=56, horizon=14):
    arr = series.to_numpy()
    X = []
    Y = []
    for start in range(len(arr) - window - horizon + 1):
        x = arr[start:start+window]
        target = np.zeros((window, horizon))
        for t in range(window):
            for h in range(horizon):
                idx = start + t + h + 1
                target[t, h] = arr[idx]
        X.append(x)
        Y.append(target)

    return np.array(X)[..., np.newaxis], np.array(Y)

X_train_seq, Y_train_seq = make_seq2seq_windows(
    rail_train,
    window=n_steps,
    horizon=ahead
)

X_valid_seq, Y_valid_seq = make_seq2seq_windows(
    rail_valid,
    window=n_steps,
    horizon=ahead
)

X_test_seq, Y_test_seq = make_seq2seq_windows(
    rail_test,
    window=n_steps,
    horizon=ahead
)

In [ ]:
X_train_seq.shape, Y_train_seq.shape

#### Custom evaluation metric function

All outputs are needed during training, but only the output at the last time step is useful at inference time for predictions and evaluation.
So even though we will rely on the MSE over all the outputs for training, we will use a custom metric for evaluation, to compute MSE only over the output at the last time step:

In [ ]:
def last_time_step_mse(Y_true, Y_pred):
    return keras.metrics.mse(Y_true[:, -1], Y_pred[:, -1])

#### TODO Model 2: Deep simple RNN with output prediction at every timestep
- Use two simple RNN layers with 20 units
- Compute the final output doing a linear read-out (use a Dense layer with no activation)
- Use the Adam optimizer with 0.01 learning rate
- Train it it for 20 epochs
- Don't forget to include your validation data
- Check the learning curves and the performance on the validation set

In [ ]:
np.random.seed(42)
tf.random.set_seed(42)

#TODO

In [ ]:
#SOLUTION
np.random.seed(42)
tf.random.set_seed(42)

model = keras.models.Sequential([
    keras.layers.Input([None,1]),
    keras.layers.SimpleRNN(20, return_sequences=True),
    keras.layers.SimpleRNN(20, return_sequences=True),
    keras.layers.TimeDistributed(keras.layers.Dense(ahead))
])

model.compile(loss="mse", optimizer=keras.optimizers.Adam(learning_rate=0.01), metrics=[last_time_step_mse])
history = model.fit(X_train_seq, Y_train_seq, epochs=20,
                    validation_data=(X_valid_seq, Y_valid_seq))

In [ ]:
# saving model stats for later
pred_multi_models['DeepRNN_seq2seq']= {'model': model, 'history': history}

In [ ]:
pd.DataFrame(history.history).plot()

In [ ]:
model.evaluate(X_valid_seq, Y_valid_seq)

##### Plot multiple forecasts with the new shaped output




**Attention** Now we have outputs at every time steps, the shape of the array of predictions is different!

In [ ]:
Y_pred_seq = model.predict(X_valid_seq)
Y_pred_seq.shape, Y_valid_seq.shape


You can reuse the previous function if you call it like this:
``` python
plot_multiple_forecasts(X_valid_seq, Y_valid_seq[:,-1,:, np.newaxis], Y_pred_seq[:,-1,:, np.newaxis])
```


In [ ]:
plot_multiple_forecasts(X_valid_seq, Y_valid_seq[:,-1,:, np.newaxis], Y_pred_seq[:,-1,:, np.newaxis])


Or you can modify the function fot the new data shape. Here is an example.

In [ ]:
def plot_multiple_forecasts_seq(X, Y, Y_pred):
    # X should be a rank 3 array with shapes [batch size, sequence length, 1]
    # Y and y_pred should also be rank 3 with shape [0, sequence lenght, 1]
    n_steps = X.shape[1]
    ahead = Y.shape[-1]
    plot_series(X[0, :, 0])
    plt.plot(np.arange(n_steps, n_steps + ahead), Y[0, -1, :], "bo-", label="Actual")
    plt.plot(np.arange(n_steps, n_steps + ahead), Y_pred[0, -1, :], "rx-", label="Forecast", markersize=10)
    plt.legend(fontsize=14)


In [ ]:
plot_multiple_forecasts_seq(X_valid_seq, Y_valid_seq, Y_pred_seq)

#### TODO Model 3: Deep LSTM with output prediction at every timestep

- Use two LSTM layers with 20 units
- Compute the final output doing a linear read-out (use a Dense layer with no activation)
- Use the Adam optimizer with the default learning rate
- Train it it for 20 epochs
- Don't forget to include your validation data
- Check the learning curves and the performance on the validation set

In [ ]:
np.random.seed(42)
tf.random.set_seed(42)

#TODO

In [ ]:
#SOLUTION
np.random.seed(42)
tf.random.set_seed(42)

model = keras.models.Sequential([
    keras.layers.LSTM(20, return_sequences=True, input_shape=[None, 1]),
    keras.layers.LSTM(20, return_sequences=True),
    keras.layers.TimeDistributed(keras.layers.Dense(ahead))
])

# Training the model

model.compile(loss="mse", optimizer=keras.optimizers.Adam(learning_rate=0.01), metrics=[last_time_step_mse])
history = model.fit(X_train_seq, Y_train_seq, epochs=20,
                    validation_data=(X_valid_seq, Y_valid_seq))


In [ ]:
# saving model stats for later
pred_multi_models['DeepRNN_LSTM']= {'model': model, 'history': history}

In [ ]:
pd.DataFrame(history.history).plot()

In [ ]:
model.evaluate(X_valid_seq, Y_valid_seq)

In [ ]:
Y_pred_seq = model.predict(X_valid_seq)
plot_multiple_forecasts(X_valid_seq, Y_valid_seq[:,-1,:, np.newaxis], Y_pred_seq[:,-1,:, np.newaxis])

#### TODO Model 4: Deep GRU with output prediction at every timestep
- Use two GRU layers with 20 units
- Compute the final output doing a linear read-out (use a Dense layer with no activation)
- Use the Adam optimizer with the default learning rate
- Train it it for 20 epochs
- Don't forget to include your validation data
- Check the learning curves and the performance on the validation set

In [ ]:
np.random.seed(42)
tf.random.set_seed(42)

#TODO

In [ ]:
#SOLUTION
np.random.seed(42)
tf.random.set_seed(42)

model = keras.models.Sequential([
    keras.layers.GRU(20, return_sequences=True, input_shape=[None, 1]),
    keras.layers.GRU(20, return_sequences=True),
    keras.layers.TimeDistributed(keras.layers.Dense(ahead))
])

# Training the model

model.compile(loss="mse", optimizer="adam", metrics=[last_time_step_mse])
history = model.fit(X_train_seq, Y_train_seq, epochs=20,
                    validation_data=(X_valid_seq, Y_valid_seq))

In [ ]:
# saving model stats for later
pred_multi_models['DeepRNN_GRU']= {'model': model, 'history': history}

In [ ]:
pd.DataFrame(history.history).plot()

In [ ]:
model.evaluate(X_valid_seq, Y_valid_seq)

In [ ]:
Y_pred_seq = model.predict(X_valid_seq)
plot_multiple_forecasts(X_valid_seq, Y_valid_seq[:,-1,:, np.newaxis], Y_pred_seq[:,-1,:, np.newaxis])

### TODO: Model comparison for prediction task 2

Create a summary table with the validation performaces of all models used in this second task. Choose the best of them and evaluate it on the test set. Did it generalize as well as predicted by the validation score?

In [ ]:
#SOLUTION
data = []
for estim, log in pred_multi_models.items():
    try:
        val_loss = log['history'].history['last_time_step_mse'][-1]
    except KeyError: 
        val_loss = log['history'].history['val_loss'][-1]

    data.append({'Model': estim, 
                'Validation loss': val_loss})
pd.DataFrame(data)

`#SOLUTION`

**Best model was the deep RNN with LSTM layers**
Here is it's performance on the test set.

In [ ]:
#TODO Complete with the best model
model = pred_multi_models[...]['model']
display(model.evaluate(X_test_seq, Y_test_seq))

Y_pred_seq = model.predict(X_test_seq)
plot_multiple_forecasts(X_test_seq, Y_test_seq[:,-1,:, np.newaxis], Y_pred_seq[:,-1,:, np.newaxis])

In [ ]:
#SOLUTION
model = pred_multi_models['DeepRNN_LSTM']['model']
display(model.evaluate(X_test_seq, Y_test_seq))

Y_pred_seq = model.predict(X_test_seq)
plot_multiple_forecasts(X_test_seq, Y_test_seq[:,-1,:, np.newaxis], Y_pred_seq[:,-1,:, np.newaxis])

## Extra models

Sequences can be treated with convolutional layers as we'll see in class soon. Here are two example models applied to the time series data we have used in this notebook.

#### Generating appropriate data and prediciton targets

In [ ]:
X_train = X_train_seq
Y_train = Y_train_seq

X_valid = X_valid_seq
Y_valid = Y_valid_seq

X_test = X_test_seq
Y_test = Y_test_seq

#### Model 5: 1D convolution + GRUs
**Attention** Due to using a convolution stride of 2, the convolutional layer subsamples the input series by a factor of 2. That means that the predicted outputs will be related to the prediction targets with a similar subsampling factor.

Additionally, the kernel size of 4 in combination with using a valid convolution makes that the first predicted output corresponds to the 4th time-steps.

This means you should compare your predictions to a cropped and subsampled version of the targets array, as follows:
```
subsampled_target = target[:, 3::2]
```

##### TODO: train the conv + GRU model
 - Use the Adam optimizer with the default learning rate
 - Train it it for 20 epochs
 - Don't forget to include your validation data
 - Check the learning curves and the performance on the validation set




In [ ]:
np.random.seed(42)
tf.random.set_seed(42)

model = keras.models.Sequential([
    keras.layers.Conv1D(filters=20, kernel_size=4, strides=2, padding="valid",
                        input_shape=[None, 1]),
    keras.layers.GRU(20, return_sequences=True),
    keras.layers.GRU(20, return_sequences=True),
    keras.layers.TimeDistributed(keras.layers.Dense(ahead))
])

#TODO train the model

In [ ]:
#SOLUTION

np.random.seed(42)
tf.random.set_seed(42)

model = keras.models.Sequential([
    keras.layers.Input([None, 1]),
    keras.layers.Conv1D(filters=20, kernel_size=4, strides=2, padding="valid"),
    keras.layers.GRU(20, return_sequences=True),
    keras.layers.GRU(20, return_sequences=True),
    keras.layers.TimeDistributed(keras.layers.Dense(ahead))
])

model.compile(loss="mse", optimizer="adam", metrics=[last_time_step_mse])
history = model.fit(X_train, Y_train[:, 3::2], epochs=20,
                    validation_data=(X_valid, Y_valid[:, 3::2]))

In [ ]:
# saving model stats for later
extra_models = {}
extra_models['DeepRNN_CNN']= {'model': model, 'history': history}

In [ ]:
pd.DataFrame(history.history).plot()

In [ ]:
model.evaluate(X_valid, Y_valid[:, 3::2])

In [ ]:
Y_pred = model.predict(X_valid)
plot_multiple_forecasts_seq(X_valid, Y_valid[:, 3::2], Y_pred)

#### Model 6: WaveNet-like model
![Wavenet diagram](http://benanne.github.io/images/wavenet.png)
[From Oord et al., 2016](https://arxiv.org/abs/1609.03499)

In [ ]:
np.random.seed(42)
tf.random.set_seed(42)

model = keras.models.Sequential()
model.add(keras.layers.InputLayer(input_shape=[None, 1]))
for rate in (1, 2, 4, 8) * 2:
    model.add(keras.layers.Conv1D(filters=20, kernel_size=2, padding="causal",
                                  activation="relu", dilation_rate=rate))
model.add(keras.layers.Conv1D(filters=ahead, kernel_size=1))
model.summary()

##### TODO: train the WaveNet model
 - Use the Adam optimizer with the default learning rate
 - Train it it for 20 epochs
 - Don't forget to include your validation data
 - Check the learning curves and the performance on the validation set

In [ ]:
#TODO

In [ ]:
#SOLUTION
model.compile(loss="mse", optimizer="adam", metrics=[last_time_step_mse])
history = model.fit(X_train, Y_train, epochs=20,
                    validation_data=(X_valid, Y_valid))

In [ ]:
# saving model stats for later
extra_models['WaveNet']= {'model': model, 'history': history}

In [ ]:
pd.DataFrame(history.history).plot()

In [ ]:
model.evaluate(X_valid, Y_valid)

In [ ]:
Y_pred = model.predict(X_valid)
plot_multiple_forecasts_seq(X_valid, Y_valid, Y_pred)

#### Result summary wiht extra models

In [ ]:
#SOLUTION
all_models = dict()
all_models.update(pred_multi_models)
all_models.update(extra_models)
data = []
for estim, log in all_models.items():
    try:
        val_loss = log['history'].history['last_time_step_mse'][-1]
    except KeyError: 
        val_loss = log['history'].history['val_loss'][-1]

    data.append({'Model': estim, 
                'Validation loss': val_loss})
pd.DataFrame(data)